# Compressible flow with friction, heat addition, and area change

**A. K. Majumdar and A. Bandyopadhyay**, *"Numerical Modeling of Compressible Flow with Friction and Heat
Transfer using the Generalized Fluid System Simulation Program (GFSSP),"* Thermal & Fluids Analysis
Workshop, TFAWS07-1016 (2007).

The paper verifies a *network* finite-volume solver against the classical one-dimensional
compressible-flow theory: Fanno flow (friction alone), Rayleigh flow (heat alone), the two combined,
and both combined again in a converging–diverging passage.
That is exactly the ground Nefes's length-bearing elements have to stand on, so the same five cases
are rebuilt here from the public catalog and compared against independently integrated references.

The question each case answers is not "do the curves overlap" but **what does the network chain
converge to, and how fast**.
Every case is therefore run at several segment counts $N$, and the observed order of accuracy is
reported alongside the profile.

| Case | Physics | Nefes assembly | Result |
| --- | --- | --- | --- |
| 1 | Fanno (friction, constant area) | `fanno_pipe` | exact classical relation, **2nd order** |
| 2 | Rayleigh (heat, constant area, frictionless) | `duct` + `heat_release_flame` chain | **exact to machine precision** |
| 3 | Friction **and** heat, constant area | `pipe` + `heat_release_flame` chain | **1st order** |
| 4 | CD nozzle, friction | `isentropic_area_change` + `pipe` chain | **1st order** |
| 5 | CD nozzle, friction and heat | Case 4 plus heat lumps | **1st order** |


In [ ]:
import math

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

import nefes
from nefes.elements import catalog as cat
from nefes.plotting import COLORWAY, use_nefes_theme

use_nefes_theme()

# Nitrogen as in the paper: R = 55.19 ft-lbf/(lbm.R), converted to SI.
R_N2 = 55.19 * 4.448221615 * 0.3048 * 9.0 / (0.45359237 * 5.0)
GAMMA = 1.4
CP = GAMMA * R_N2 / (GAMMA - 1.0)
GAS = nefes.perfect_gas(R_N2, GAMMA)

INCH = 0.0254
PSI = 6894.757293168
BTU_PER_S = 1055.05585262

# The paper's constant-area duct (Cases 1-3) and its inlet state.
D_PIPE = 6.0 * INCH
L_PAPER = 3207.0 * INCH
A_PIPE = math.pi * D_PIPE**2 / 4.0
P_IN = 50.0 * PSI
T_IN = (80.0 - 32.0) * 5.0 / 9.0 + 273.15

print(f"nitrogen: R = {R_N2:.2f} J/(kg K), cp = {CP:.1f} J/(kg K)")
print(f"duct:     L = {L_PAPER:.2f} m, D = {D_PIPE:.4f} m, A = {A_PIPE:.6f} m^2")
print(f"inlet:    p = {P_IN / 1e3:.1f} kPa, T = {T_IN:.2f} K")

## The references

Two independent oracles are used, neither of which calls Nefes.

**Classical Fanno relations (Case 1).**
For adiabatic constant-area flow with wall friction, the duct length between two Mach numbers is
fixed by

$$
\mathcal F(M)
=
\frac{1-M^{2}}{\gamma M^{2}}
+
\frac{\gamma+1}{2\gamma}
\ln\!\left[\frac{(\gamma+1)M^{2}}{2+(\gamma-1)M^{2}}\right],
\qquad
\frac{f\,\Delta x}{D} = \mathcal F(M_{1}) - \mathcal F(M_{2}),
$$

with the static ratios $p/p^{*}$ and $T/T^{*}$ following from $M$ alone.
Inverting $\mathcal F$ at each station with a bracketing root solver gives the exact profile.

**Generalized one-dimensional flow (Cases 2–5).**
With area change, friction, and heat addition acting together the paper integrates

$$
\frac{\mathrm{d}M}{\mathrm{d}x}
=
\frac{M}{2}\,
\frac{1+\tfrac{\gamma-1}{2}M^{2}}{1-M^{2}}
\left[
\gamma M^{2}\frac{f}{D}
+\left(1+\gamma M^{2}\right)\frac{1}{T_{t}}\frac{\mathrm{d}T_{t}}{\mathrm{d}x}
-\frac{2}{A}\frac{\mathrm{d}A}{\mathrm{d}x}
\right],
$$

where $\mathrm{d}T_{t}/\mathrm{d}x = \dot Q/(\dot m\,c_{p}L)$ for heat spread uniformly over the
passage.
Static temperature follows from $T_{t}$ and $M$, and static pressure from mass conservation at the
local area.
Fanno and Rayleigh are its $\dot Q = 0$ and $f = 0$ specializations.

The $(1-M^{2})$ denominator makes this system **singular at $M = 1$**, which matters below: two of
the paper's cases choke exactly at the duct exit, and no reference integrated through that point is
trustworthy there.

In [ ]:
def fanno_length_function(M):
    """The classical Fanno parameter F(M) = f L*/D for a perfect gas."""
    return (1.0 - M**2) / (GAMMA * M**2) + (GAMMA + 1.0) / (2.0 * GAMMA) * math.log(
        (GAMMA + 1.0) * M**2 / (2.0 + (GAMMA - 1.0) * M**2)
    )


def fanno_ratios(M):
    """Static (p/p*, T/T*) of a Fanno state at Mach M."""
    return (
        math.sqrt((GAMMA + 1.0) / (2.0 + (GAMMA - 1.0) * M**2)) / M,
        (GAMMA + 1.0) / (2.0 + (GAMMA - 1.0) * M**2),
    )


def fanno_reference(x, M1, p1, T1, friction, diameter):
    """Exact Fanno profile at stations x, by inverting F(M) station by station."""
    M = np.array(
        [
            brentq(
                lambda m: fanno_length_function(m) - (fanno_length_function(M1) - friction * xi / diameter),
                M1,
                0.9999,
            )
            for xi in x
        ]
    )
    p_ratio, T_ratio = np.array([fanno_ratios(m) for m in M]).T
    p1_ratio, T1_ratio = fanno_ratios(M1)
    return {"x": x, "M": M, "p": p1 * p_ratio / p1_ratio, "T": T1 * T_ratio / T1_ratio}


def shapiro_reference(M1, length, *, friction=0.0, heat=0.0, area=None, diameter=None, n_eval=801):
    """Integrate the generalized one-dimensional flow equation; return the dense profile."""
    area = area if area is not None else (lambda _x: A_PIPE)
    diameter = diameter if diameter is not None else (lambda _x: D_PIPE)
    Tt1 = T_IN * (1.0 + 0.5 * (GAMMA - 1.0) * M1**2)
    mdot = P_IN * M1 * area(0.0) * math.sqrt(GAMMA / (R_N2 * T_IN))
    dTt_dx = heat / (mdot * CP * length) if heat else 0.0
    step = 1.0e-6 * length

    def rhs(x, y):
        M, Tt = float(y[0]), float(y[1])
        lo, hi = max(x - step, 0.0), min(x + step, length)
        dA_dx = (area(hi) - area(lo)) / (hi - lo)
        bracket = GAMMA * M**2 * friction / diameter(x) + (1.0 + GAMMA * M**2) * dTt_dx / Tt - 2.0 * dA_dx / area(x)
        return [0.5 * M * (1.0 + 0.5 * (GAMMA - 1.0) * M**2) / (1.0 - M**2) * bracket, dTt_dx]

    ivp = solve_ivp(
        rhs, (0.0, length), [M1, Tt1], rtol=1.0e-10, atol=1.0e-12, dense_output=True, max_step=length / 2000.0
    )
    if not ivp.success:
        raise RuntimeError(f"reference integration failed: {ivp.message}")
    x = np.linspace(0.0, length, n_eval)
    M, Tt = np.array([ivp.sol(xi) for xi in x]).T
    T = Tt / (1.0 + 0.5 * (GAMMA - 1.0) * M**2)
    A = np.array([area(xi) for xi in x])
    return {
        "x": x,
        "M": M,
        "T": T,
        "p": mdot / (M * A) * np.sqrt(R_N2 * T / GAMMA),
        "mdot": mdot,
        "Tt": Tt1,
    }

### How each case is driven

Every network below is driven the same way: a **prescribed inlet mass flow and total temperature**,
and a **prescribed outlet static pressure** taken from the reference at the last station.
Pinning $\dot m$ makes the comparison like-for-like, because the reference fixes the inlet Mach by
construction; the interior profile and the inlet pressure the friction demands are then genuine
predictions of the network rather than restatements of a boundary condition.


In [ ]:
PANELS = (("M", "Mach"), ("p", "static pressure  [kPa]"), ("T", "static temperature  [K]"))
SCALE = {"M": 1.0, "p": 1.0e-3, "T": 1.0}


def compare_figure(reference, series, title, length):
    """Overlay the reference profile and one or more Nefes chains over x / L."""
    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.095)
    for col, (key, ylabel) in enumerate(PANELS, start=1):
        fig.add_trace(
            go.Scatter(
                x=reference["x"] / length,
                y=reference[key] * SCALE[key],
                name="reference",
                legendgroup="reference",
                showlegend=(col == 1),
                mode="lines",
                line=dict(width=3, color=COLORWAY[0]),
            ),
            row=1,
            col=col,
        )
        for j, (label, x, profile) in enumerate(series):
            stride = max(1, len(x) // 30)  # keep the markers readable on a fine chain
            fig.add_trace(
                go.Scatter(
                    x=x[::stride] / length,
                    y=profile[key][::stride] * SCALE[key],
                    name=label,
                    legendgroup=label,
                    showlegend=(col == 1),
                    mode="markers",
                    marker=dict(size=5, symbol=["circle-open", "x"][j % 2], color=COLORWAY[j + 1]),
                ),
                row=1,
                col=col,
            )
        fig.update_xaxes(title_text="x / L", row=1, col=col)
        fig.update_yaxes(title_text=ylabel, row=1, col=col)
    fig.update_layout(
        title=dict(text=title, y=0.96, yanchor="top"),
        height=460,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0),
        margin=dict(t=110, b=70, l=70, r=30),
    )
    return fig


def profile_errors(x, profile, reference):
    """Largest relative deviation from the reference, interpolated onto the Nefes stations."""
    return {
        key: float(
            np.max(
                np.abs(profile[key] - np.interp(x, reference["x"], reference[key]))
                / np.abs(np.interp(x, reference["x"], reference[key]))
            )
        )
        for key, _ in PANELS
    }


def report(label, errors):
    body = "   ".join(f"{key} {value:9.2e}" for key, value in errors.items())
    print(f"  {label:<24s} max relative error:   {body}")


def observed_order(errors, counts):
    """Least-squares slope of log(error) against log(N)."""
    return -float(np.polyfit(np.log(counts), np.log(errors), 1)[0])


def refinement_figure(runs, title):
    """Error against segment count on log-log axes, with first- and second-order guides."""
    fig = go.Figure()
    for j, (label, counts, errors) in enumerate(runs):
        order = observed_order(errors, counts)
        tag = f"order {order:.2f}" if order > 0.2 else "does not converge"
        fig.add_trace(
            go.Scatter(
                x=counts,
                y=errors,
                name=f"{label}  ({tag})",
                mode="markers+lines",
                marker=dict(size=9),
                line=dict(width=2, color=COLORWAY[j]),
            )
        )
    counts = np.array(runs[-1][1], dtype=float)
    anchor = runs[-1][2][0]
    for power, dash in ((1.0, "dot"), (2.0, "dash")):
        fig.add_trace(
            go.Scatter(
                x=counts,
                y=anchor * (counts[0] / counts) ** power,
                name=f"order {power:.0f}",
                mode="lines",
                line=dict(width=1.5, dash=dash, color="grey"),
                hoverinfo="skip",
            )
        )
    fig.update_layout(
        title=dict(text=title, y=0.95, yanchor="top"),
        xaxis=dict(
            title="segments N",
            type="log",
            tickmode="array",
            tickvals=list(counts),
            ticktext=[f"{int(n)}" for n in counts],
        ),
        yaxis=dict(title="max relative error in M", type="log", dtick=1, tickformat=".0e"),
        width=820,
        height=500,
        legend=dict(yanchor="bottom", y=0.02, xanchor="left", x=0.02),
        margin=dict(t=90, b=60, l=90, r=30),
    )
    return fig

## Case 1 — Fanno flow

Wall friction alone, at constant area.
The duct length is set by the exact relation for an inlet $M_{1}=0.5$ and a clearly subsonic exit
$M_{2}=0.85$, so the comparison stays well away from the sonic singularity and the classical
relations apply without qualification.

Both pipe closures are run on the identical network.
`darcy-weisbach` is the lumped total-pressure head $\Delta p_t = (fL/D)\,\tfrac12\varrho u^{2}$,
correct in the incompressible limit; `momentum` is the compressible segment balance.
The gap between them at $M \sim 0.5$–$0.85$ is the compressibility the lumped head omits, and it is
the reason `fanno_pipe` defaults to the momentum closure.

In [ ]:
M1_FANNO, M2_FANNO, F_FANNO = 0.5, 0.85, 0.002
N_FANNO = 64

L_FANNO = D_PIPE / F_FANNO * (fanno_length_function(M1_FANNO) - fanno_length_function(M2_FANNO))
TT_FANNO = T_IN * (1.0 + 0.5 * (GAMMA - 1.0) * M1_FANNO**2)
MDOT_FANNO = P_IN * M1_FANNO * A_PIPE * math.sqrt(GAMMA / (R_N2 * T_IN))
P_OUT_FANNO = P_IN * fanno_ratios(M2_FANNO)[0] / fanno_ratios(M1_FANNO)[0]

print(f"Fanno duct: L = {L_FANNO:.3f} m for M {M1_FANNO} -> {M2_FANNO} at f = {F_FANNO}")
print(f"            mdot = {MDOT_FANNO:.4f} kg/s, p_out = {P_OUT_FANNO / 1e3:.2f} kPa")


def solve_fanno(n_segments, formulation):
    """inlet -> fanno_pipe(N segments) -> outlet, sampled at every segment face."""
    network = nefes.Network(
        GAS,
        nodes=[
            cat.mass_flow_inlet(MDOT_FANNO, TT_FANNO, name="in"),
            cat.fanno_pipe(L_FANNO, D_PIPE, F_FANNO, n_segments, formulation=formulation, name="duct"),
            cat.pressure_outlet(P_OUT_FANNO, TT_FANNO, name="out"),
        ],
        edges=[(0, 1, A_PIPE), (1, 2, A_PIPE)],
    )
    sol = network.solve()
    assert sol.converged, f"N = {n_segments} did not converge (residual {sol.residual_norm:.2e})"
    sol.verify()
    faces = [0] + list(sol.composite("duct").internal_edges) + [1]
    x = np.linspace(0.0, L_FANNO, n_segments + 1)
    return x, {key: np.array([sol.edge(e)[key] for e in faces]) for key, _ in PANELS}


x_exact = np.linspace(0.0, L_FANNO, 401)
reference_1 = fanno_reference(x_exact, M1_FANNO, P_IN, T_IN, F_FANNO, D_PIPE)

print("\nCase 1 - Fanno flow")
series_1 = []
for formulation in ("darcy-weisbach", "momentum"):
    x, profile = solve_fanno(N_FANNO, formulation)
    series_1.append((formulation, x, profile))
    report(formulation, profile_errors(x, profile, reference_1))

compare_figure(reference_1, series_1, f"Case 1 - Fanno flow, N = {N_FANNO}", L_FANNO).show()

The momentum chain sits on the classical curve; the lumped Darcy head does not.
Refining $N$ settles whether that gap is a resolution artefact or a difference of closure.

In [ ]:
COUNTS_FANNO = (8, 16, 32, 64, 128)

runs_1 = []
for formulation in ("darcy-weisbach", "momentum"):
    errors = []
    for n in COUNTS_FANNO:
        x, profile = solve_fanno(n, formulation)
        errors.append(profile_errors(x, profile, fanno_reference(x, M1_FANNO, P_IN, T_IN, F_FANNO, D_PIPE))["M"])
    runs_1.append((formulation, COUNTS_FANNO, errors))
    print(f"  {formulation:<16s} " + "  ".join(f"N={n}: {e:.2e}" for n, e in zip(COUNTS_FANNO, errors)))

refinement_figure(runs_1, "Case 1 - convergence onto the classical Fanno profile").show()

## Cases 2 and 3 — heat addition, with and without friction

The paper's heated cases are tuned to choke the duct exactly at the exit ($M \to 1$ at $x = L$).
That is a legitimate operating point, but it is a singular point of the reference: the
$(1-M^{2})$ denominator makes the integrated profile lose all accuracy in the last few per cent of
the duct, and the outlet pressure read from it there is meaningless as a boundary condition.

The comparison is therefore run over the first **90 %** of the paper's duct, at the paper's inlet
Mach, duct diameter, and volumetric heat rate.
That keeps the exit at $M \approx 0.8$, where the reference is well conditioned, and leaves the
physics being tested untouched.
Nefes itself solves the full choked duct without difficulty; it is the *reference* that cannot be
trusted through $M = 1$.

Distributed heating is a chain of $N$ segments, each a lossless `duct` (Case 2) or a frictional
`pipe` (Case 3) followed by a compact `heat_release_flame` carrying $\dot Q/N$.

In [ ]:
DOMAIN_FRACTION = 0.9
L_HEATED = DOMAIN_FRACTION * L_PAPER


def solve_heated_chain(n_segments, *, friction, heat, reference):
    """inlet -> (duct|pipe, flame) x N -> outlet, sampled downstream of every heat lump."""
    nodes = [cat.mass_flow_inlet(reference["mdot"], reference["Tt"], name="in")]
    edges = []
    node = 0
    for i in range(n_segments):
        if friction:
            nodes.append(cat.pipe(L_HEATED / n_segments, D_PIPE, friction, formulation="momentum", name=f"p{i}"))
        else:
            nodes.append(cat.duct(L_HEATED / n_segments, name=f"d{i}"))
        edges.append((node, node + 1, A_PIPE))
        node += 1
        nodes.append(cat.heat_release_flame(heat / n_segments, name=f"q{i}"))
        edges.append((node, node + 1, A_PIPE))
        node += 1
    nodes.append(cat.pressure_outlet(float(reference["p"][-1]), reference["Tt"], name="out"))
    edges.append((node, node + 1, A_PIPE))

    sol = nefes.Network(GAS, nodes=nodes, edges=edges).solve()
    assert sol.converged, f"N = {n_segments} did not converge (residual {sol.residual_norm:.2e})"
    sol.verify()
    # the edge downstream of flame i is 2i + 2; for the last flame that is the outlet edge
    faces = [0] + [2 * i + 2 for i in range(n_segments - 1)] + [len(edges) - 1]
    x = np.linspace(0.0, L_HEATED, n_segments + 1)
    return x, {key: np.array([sol.edge(e)[key] for e in faces]) for key, _ in PANELS}


CASES_HEATED = {
    "Case 2 - Rayleigh flow": dict(M1=0.46, friction=0.0, heat=2088.0 * BTU_PER_S * DOMAIN_FRACTION),
    "Case 3 - friction and heat": dict(M1=0.45, friction=0.002, heat=555.0 * BTU_PER_S * DOMAIN_FRACTION),
}
COUNTS_HEATED = (20, 40, 80, 160)

runs_23 = []
for title, spec in CASES_HEATED.items():
    reference = shapiro_reference(spec["M1"], L_HEATED, friction=spec["friction"], heat=spec["heat"])
    print(f"\n{title}   (exit of the compared domain at M = {reference['M'][-1]:.3f})")
    errors = []
    for n in COUNTS_HEATED:
        x, profile = solve_heated_chain(n, friction=spec["friction"], heat=spec["heat"], reference=reference)
        errors.append(profile_errors(x, profile, reference)["M"])
        if n == COUNTS_HEATED[1]:
            series = [(f"Nefes chain, N = {n}", x, profile)]
    runs_23.append((title.split(" - ")[1], COUNTS_HEATED, errors))
    print("  " + "  ".join(f"N={n}: {e:.2e}" for n, e in zip(COUNTS_HEATED, errors)))
    compare_figure(reference, series, f"{title}, N = {COUNTS_HEATED[1]}", L_HEATED).show()

**Rayleigh flow is reproduced to solver tolerance**, not merely closely: the error sits near
$10^{-9}$ and does not improve with $N$ because there is nothing left to resolve.
The reason is structural.
For constant-area flow the `heat_release_flame` kernel enforces the exact thin-flame jump
$\varrho u^{2} + p = \text{const}$ together with the total-enthalpy rise $\dot Q/\dot m$, and that
jump *is* the Rayleigh relation.
A chain of lossless ducts and compact flames therefore does not approximate Rayleigh flow — it
integrates it exactly, whatever the segment count.

Adding friction breaks that exactness, and Case 3 converges at **first order**.
Each segment applies its wall friction over the segment and then its heat at a point, so the two
effects are staggered rather than simultaneous; the resulting splitting error is $O(\Delta x)$.

In [ ]:
refinement_figure(runs_23, "Cases 2 and 3 - convergence of the heated chains").show()

## Cases 4 and 5 — converging–diverging nozzle

The paper's nozzle runs 9 in at inlet to a 6 in throat and out to a 12 in exit, with the diameter
varying linearly on each side, at $M_{1}=0.25$ and a deliberately large Darcy $f = 0.05$.
Its axial length is not stated numerically in the text, so $L = 100$ in with the throat at
mid-length is used here; at $f = 0.05$ that keeps the whole passage subsonic (peak $M \approx 0.87$
at the throat), which is the regime the paper plots and the regime Nefes presently covers.
Case 5 adds a uniform $\dot Q = 100$ Btu/s, likewise not given in the paper extract, chosen so both
the reference and the network stay clear of the throat going sonic.

Each segment is an `isentropic_area_change` into the downstream area followed by a `pipe` carrying
that segment's length and local hydraulic diameter, with Case 5 inserting a heat lump after the
pipe.

In [ ]:
D_NOZZLE_IN, D_THROAT, D_NOZZLE_OUT = 9.0 * INCH, 6.0 * INCH, 12.0 * INCH
L_NOZZLE = 100.0 * INCH
X_THROAT = 0.5 * L_NOZZLE
F_NOZZLE = 0.05
M1_NOZZLE = 0.25
Q_NOZZLE = 100.0 * BTU_PER_S


def nozzle_diameter(x):
    if x <= X_THROAT:
        return D_NOZZLE_IN + (D_THROAT - D_NOZZLE_IN) * (x / X_THROAT)
    return D_THROAT + (D_NOZZLE_OUT - D_THROAT) * ((x - X_THROAT) / (L_NOZZLE - X_THROAT))


def nozzle_area(x):
    return math.pi * nozzle_diameter(x) ** 2 / 4.0


def solve_nozzle_chain(n_segments, *, heat, reference):
    """inlet -> (area change, pipe[, flame]) x N -> outlet, sampled at every segment exit."""
    x = np.linspace(0.0, L_NOZZLE, n_segments + 1)
    areas = [nozzle_area(xi) for xi in x]
    nodes = [cat.mass_flow_inlet(reference["mdot"], reference["Tt"], name="in")]
    edges = []
    node, faces = 0, []
    for i in range(n_segments):
        nodes.append(cat.isentropic_area_change(name=f"a{i}"))
        edges.append((node, node + 1, areas[i]))
        node += 1
        nodes.append(
            cat.pipe(float(x[i + 1] - x[i]), nozzle_diameter(x[i + 1]), F_NOZZLE, formulation="momentum", name=f"p{i}")
        )
        edges.append((node, node + 1, areas[i + 1]))
        node += 1
        if heat:
            nodes.append(cat.heat_release_flame(heat / n_segments, name=f"q{i}"))
            edges.append((node, node + 1, areas[i + 1]))
            node += 1
        faces.append(len(edges))  # the edge opened by the next segment, or the outlet edge
    nodes.append(cat.pressure_outlet(float(reference["p"][-1]), reference["Tt"], name="out"))
    edges.append((node, node + 1, areas[-1]))

    sol = nefes.Network(GAS, nodes=nodes, edges=edges).solve()
    assert sol.converged, f"N = {n_segments} did not converge (residual {sol.residual_norm:.2e})"
    sol.verify()
    return x, {key: np.array([sol.edge(0)[key]] + [sol.edge(e)[key] for e in faces]) for key, _ in PANELS}


CASES_NOZZLE = {
    "Case 4 - nozzle with friction": 0.0,
    "Case 5 - nozzle with friction and heat": Q_NOZZLE,
}
COUNTS_NOZZLE = (32, 64, 128, 256)

runs_45 = []
for title, heat in CASES_NOZZLE.items():
    reference = shapiro_reference(
        M1_NOZZLE, L_NOZZLE, friction=F_NOZZLE, heat=heat, area=nozzle_area, diameter=nozzle_diameter
    )
    print(f"\n{title}   (reference peaks at M = {reference['M'].max():.3f})")
    errors = []
    for n in COUNTS_NOZZLE:
        x, profile = solve_nozzle_chain(n, heat=heat, reference=reference)
        errors.append(profile_errors(x, profile, reference)["M"])
        if n == COUNTS_NOZZLE[-1]:
            series = [(f"Nefes chain, N = {n}", x, profile)]
    runs_45.append((title.split(" - ")[1], COUNTS_NOZZLE, errors))
    print("  " + "  ".join(f"N={n}: {e:.2e}" for n, e in zip(COUNTS_NOZZLE, errors)))
    compare_figure(reference, series, f"{title}, N = {COUNTS_NOZZLE[-1]}", L_NOZZLE).show()

refinement_figure(runs_45, "Cases 4 and 5 - convergence of the nozzle chains").show()

## What the five cases establish

| Case | Physics | Observed order | Error at the finest grid |
| --- | --- | --- | --- |
| 1, momentum | Fanno | 2 | $\sim 6\times10^{-6}$ at $N = 128$ |
| 1, Darcy–Weisbach | Fanno | — (converges to a different answer) | $\sim 7\times10^{-2}$ |
| 2 | Rayleigh | exact | $\sim 3\times10^{-9}$, independent of $N$ |
| 3 | friction and heat | 1 | $\sim 1\times10^{-4}$ at $N = 160$ |
| 4 | nozzle, friction | 1 | $\sim 4\times10^{-3}$ at $N = 256$ |
| 5 | nozzle, friction and heat | 1 | $\sim 8\times10^{-3}$ at $N = 256$ |

Three conclusions carry beyond this benchmark.

**The element that carries an exact jump gives an exact answer.**
The compact flame's constant-area momentum relation is the Rayleigh relation, so Case 2 is limited
only by the Newton tolerance.
Nothing about the segment count enters.

**The friction pipe is second order, and the closure matters.**
The momentum pipe integrates its wall head across the segment rather than lumping it, which is why
Case 1 converges quadratically.
The Darcy–Weisbach head is not a coarse version of it: it is the low-Mach limit, and at
$M \gtrsim 0.5$ it converges confidently onto a different profile.
Choosing between them is a modelling decision, not a resolution one — which is why the atomic
`pipe` keeps the Darcy head that hydraulic network practice expects while `fanno_pipe` defaults to
the momentum closure.

**Splitting two effects across a segment costs an order.**
Wherever a segment applies friction over its length and then heat, or an area change and then
friction, the staggering is an $O(\Delta x)$ error and the chain converges at first order.
That is the same rate the `tapered_duct` composite shows against the Webster horn, and it sets the
segment count a distributed model needs: reaching a given tolerance costs $1/\varepsilon$ segments
in the split cases against $1/\sqrt{\varepsilon}$ for pure Fanno.